In [1]:
import glob
import math
import numpy as np
import polars as pl
import pandas as pd
from tqdm import tqdm

In [2]:
# Download list of Olink genes

!dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/olink/preprocessed/proteomics_genes.txt

olink_genes = pl.read_csv('proteomics_genes.txt', has_header=False).rename({'column_1': 'region'})
olink_genes

Error: path
"/home/dnanexus/ukbgym/utils/average_pheno_per_variant/proteomics_genes.txt"
already exists but -f/--overwrite was not set


region
str
"""ENSG00000266967"""
"""ENSG00000114779"""
"""ENSG00000097007"""
"""ENSG00000060971"""
"""ENSG00000157766"""
…
"""ENSG00000173465"""
"""ENSG00000105428"""
"""ENSG00000188372"""


In [3]:
# Download annotation file for variant-gene mapping
!dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/wgs/qced_maf1e-3_loftee_olink_genes/qced_maf1e-3_loftee_olink_genes_EURunrelated/annotations_fill_na.parquet -o /home/dnanexus/data_dir/

anno = pl.read_parquet(
    '/home/dnanexus/data_dir/annotations_fill_na.parquet', 
    columns=['id', 'region']
)

anno

Error: path "/home/dnanexus/data_dir/annotations_fill_na.parquet" already
exists but -f/--overwrite was not set


id,region
str,str
"""chr18:56603287:T:G""","""ENSG00000091164"""
"""chr18:56611517:A:G""","""ENSG00000091164"""
"""chr19:34340475:G:A""","""ENSG00000166398"""
"""chr19:5256210:T:C""","""ENSG00000105426"""
"""chr19:5257316:G:C""","""ENSG00000105426"""
…,…
"""chr16:179290:T:A""","""ENSG00000086506"""
"""chr7:148348869:A:G""","""ENSG00000174469"""
"""chr5:138680487:A:G""","""ENSG00000044115"""


In [4]:
anno['region'].value_counts(sort=True).join(olink_genes, on='region', how='semi')

region,count
str,u64
"""ENSG00000174469""",605106
"""ENSG00000185008""",444475
"""ENSG00000189283""",435466
"""ENSG00000021645""",385076
"""ENSG00000149972""",376497
…,…
"""ENSG00000179889""",1616
"""ENSG00000267368""",611
"""ENSG00000160221""",564


In [5]:
# Download Olink: covariates and PRS corrected
!dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/olink/adjusted/cauc_cov_regression_90pcs_prs.parquet -o /home/dnanexus/data_dir/olink/

phenos = pl.read_parquet('/home/dnanexus/data_dir/olink/cauc_cov_regression_90pcs_prs.parquet')

olink_genes_w_data = list(set(phenos.columns).intersection(set(olink_genes['region'])).intersection(set(anno['region'])))

phenos = phenos.select(['sample'] + olink_genes_w_data)

long_phenos = (
    phenos
    .unpivot(
        index='sample',
        on=olink_genes_w_data,
        variable_name='phenotype',
        value_name='pheno_value'
    )

    .with_columns(
        phenotype = pl.col('phenotype') + '_olink'
    )
    .drop_nulls()
)

print(long_phenos['phenotype'].value_counts(sort=True))
long_phenos

Error: path
"/home/dnanexus/data_dir/olink/cauc_cov_regression_90pcs_prs.parquet" already
exists but -f/--overwrite was not set
shape: (2_661, 2)
┌───────────────────────┬───────┐
│ phenotype             ┆ count │
│ ---                   ┆ ---   │
│ str                   ┆ u64   │
╞═══════════════════════╪═══════╡
│ ENSG00000275718_olink ┆ 39208 │
│ ENSG00000050165_olink ┆ 39208 │
│ ENSG00000108578_olink ┆ 39208 │
│ ENSG00000172016_olink ┆ 39208 │
│ ENSG00000172023_olink ┆ 39208 │
│ …                     ┆ …     │
│ ENSG00000102837_olink ┆ 31597 │
│ ENSG00000131050_olink ┆ 31502 │
│ ENSG00000111405_olink ┆ 30953 │
│ ENSG00000163131_olink ┆ 30432 │
│ ENSG00000170373_olink ┆ 29106 │
└───────────────────────┴───────┘


sample,phenotype,pheno_value
str,str,f64
"""5645319""","""ENSG00000106278_olink""",0.480639
"""5959139""","""ENSG00000106278_olink""",-1.73118
"""5732867""","""ENSG00000106278_olink""",-0.743943
"""2074480""","""ENSG00000106278_olink""",-0.627407
"""4659532""","""ENSG00000106278_olink""",0.646729
…,…,…
"""1807196""","""ENSG00000125735_olink""",1.685035
"""4223555""","""ENSG00000125735_olink""",-0.771043
"""2992773""","""ENSG00000125735_olink""",1.185389


In [6]:
# Download genotype (long gt) file
!dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/wgs/qced_maf1e-3_loftee_olink_genes/qced_maf1e-3_loftee_olink_genes_EURunrelated/gt_long.parquet -o /home/dnanexus/data_dir/

long_gt = (
    pl.scan_parquet('/home/dnanexus/data_dir/gt_long.parquet')
    .select(['id', 'sample', 'gt'])
    .filter(pl.col('gt')==1)
)
long_gt.head().collect()

Error: path "/home/dnanexus/data_dir/gt_long.parquet" already exists but
-f/--overwrite was not set


id,sample,gt
str,str,i8
"""chr10:20020006:C:T""","""5317620""",1
"""chr10:20020007:TTTTCTTGC:T""","""5546988""",1
"""chr10:20020010:T:C""","""2793793""",1
"""chr10:20020014:G:A""","""1722739""",1
"""chr10:20020014:G:A""","""1139673""",1


In [ ]:
output_dir = '/home/dnanexus/data_dir/appv_phenos/'
!mkdir -p {output_dir}

CHUNK_SIZE = 100
total_genes = len(olink_genes_w_data)
num_chunks = math.ceil(total_genes / CHUNK_SIZE)

print(f"Processing {total_genes} genes in {num_chunks} chunks...")

# Process in Batches
for i in tqdm(range(0, total_genes, CHUNK_SIZE)):
    # 1. Define the current batch of genes
    chunk_genes = olink_genes_w_data[i : i + CHUNK_SIZE]
    chunk_phenos = [f"{g}_olink" for g in chunk_genes]

# for olink_gene in tqdm(olink_genes_w_data):
    print(f"Processing chunk starting at index: {i}")
    
    (
        anno.filter(pl.col('region').is_in(chunk_genes))
        .lazy()
        .join(
            long_gt,
            on='id',
            how='inner'
        )

        .join(
            long_phenos.filter(pl.col('phenotype').is_in(chunk_phenos)).lazy(),
            on='sample',
            # on=['sample', 'phenotype'],
            how='inner'
        )

        .group_by(['id', 'region', 'phenotype'])
        .agg(
            n_individuals = pl.len().cast(pl.Int32),
            mean_pheno_value = pl.col('pheno_value').mean().cast(pl.Float32),
            std_pheno_value = pl.col('pheno_value').std().cast(pl.Float32),
        )

        .with_columns(
            mean_pheno_value_rank=pl.col('mean_pheno_value')
                .rank(method="max")
                .over("region")
                .cast(pl.Float32)
        )
        .with_columns(
            mean_pheno_value_ptile=(
                pl.col('mean_pheno_value_rank') / pl.len()
            ).cast(pl.Float32)
        )

        .sink_parquet(f'{output_dir}/olink_all_genes_EURunrelated_appv_chunk{i}.parquet')
        # .collect(engine='streaming')
    )

Processing 2661 genes in 27 chunks...


  0%|          | 0/27 [00:00<?, ?it/s]

Processing chunk starting at index: 0


  4%|▎         | 1/27 [00:45<19:34, 45.17s/it]

Processing chunk starting at index: 100


...

 93%|█████████▎| 25/27 [17:03<01:26, 43.37s/it]

Processing chunk starting at index: 2500


 96%|█████████▋| 26/27 [17:39<00:41, 41.16s/it]

Processing chunk starting at index: 2600


100%|██████████| 27/27 [18:08<00:00, 40.31s/it]


## Consolidate parquet

In [8]:
pl.read_parquet(f'{output_dir}/olink_all_genes_EURunrelated_appv_chunk0.parquet')

id,region,phenotype,n_individuals,mean_pheno_value,std_pheno_value,mean_pheno_value_rank,mean_pheno_value_ptile
str,str,str,i32,f32,f32,f32,f32
"""chr1:168580722:T:A""","""ENSG00000143184""","""ENSG00000143799_olink""",1,1.125455,null,119102.0,0.00169
"""chr1:168580722:T:A""","""ENSG00000143184""","""ENSG00000163586_olink""",1,0.499719,null,98591.0,0.001399
"""chr1:168580722:T:A""","""ENSG00000143184""","""ENSG00000271605_olink""",1,-1.363557,null,7064.0,0.0001
"""chr1:168580722:T:A""","""ENSG00000143184""","""ENSG00000120708_olink""",1,0.145929,null,76941.0,0.001092
"""chr1:168580722:T:A""","""ENSG00000143184""","""ENSG00000099365_olink""",1,0.273486,null,85950.0,0.00122
…,…,…,…,…,…,…,…
"""chr3:76192346:A:G""","""ENSG00000185008""","""ENSG00000146232_olink""",23,-0.092392,0.867017,6.099152e6,0.086567
"""chr3:76192346:A:G""","""ENSG00000185008""","""ENSG00000136698_olink""",23,0.078176,0.759473,7.557065e6,0.107259
"""chr3:76192346:A:G""","""ENSG00000185008""","""ENSG00000174721_olink""",20,-0.414718,0.942751,3.816307e6,0.054166


In [9]:
combined_output_file = "/home/dnanexus/data_dir/olink_all_genes_EURunrelated_appv_percentiles.parquet"

# 1. Get list of files manually
files = glob.glob(output_dir)
print(f"Found {len(files)} files.")

# 2. Create a list of LazyFrames
lfs = [pl.scan_parquet(f) for f in files]

# 3. Concatenate with relaxation
combined_lazy = pl.concat(lfs, how="vertical_relaxed")

# 4. Stream to disk
print("Streaming to disk...")
combined_lazy.sink_parquet(combined_output_file, engine='streaming')
print("Done.")

Found 1 files.
Streaming to disk...


Done.


In [21]:
!dx upload {combined_output_file} --path project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/ukbgym/avg_pheno_per_var/

[===========================================================>] Uploaded 24,103,679,891 of 24,103,679,891 bytes (100%) /home/dnanexus/data_dir/olink_all_genes_EURunrelated_appv_percentiles.parquet=======================================================>    ] Uploaded 22,347,251,712 of 24,103,679,891 bytes (93%) /home/dnanexus/data_dir/olink_all_genes_EURunrelated_appv_percentiles.parquet=======>                                                   ] Uploaded 3,724,541,952 of 24,103,679,891 bytes (15%) /home/dnanexus/data_dir/olink_all_genes_EURunrelated_appv_percentiles.parquet[============>                                               ] Uploaded 5,133,828,096 of 24,103,679,891 bytes (21%) /home/dnanexus/data_dir/olink_all_genes_EURunrelated_appv_percentiles.parquet[====================>                                       ] Uploaded 8,254,390,272 of 24,103,679,891 bytes (34%) /home/dnanexus/data_dir/olink_all_genes_EURunrelated_appv_percentiles.parquet[======================>           

In [18]:
a = pl.scan_parquet(combined_output_file)
a.head().collect()

id,region,phenotype,n_individuals,mean_pheno_value,std_pheno_value,mean_pheno_value_rank,mean_pheno_value_ptile
str,str,str,i32,f32,f32,f32,f32
"""chr1:168580722:T:A""","""ENSG00000143184""","""ENSG00000143799_olink""",1,1.125455,null,119102.0,0.00169
"""chr1:168580722:T:A""","""ENSG00000143184""","""ENSG00000163586_olink""",1,0.499719,null,98591.0,0.001399
"""chr1:168580722:T:A""","""ENSG00000143184""","""ENSG00000271605_olink""",1,-1.363557,null,7064.0,0.0001
"""chr1:168580722:T:A""","""ENSG00000143184""","""ENSG00000120708_olink""",1,0.145929,null,76941.0,0.001092
"""chr1:168580722:T:A""","""ENSG00000143184""","""ENSG00000099365_olink""",1,0.273486,null,85950.0,0.00122


In [20]:
a.select(['region']).collect()['region'].unique()

region
str
"""ENSG00000129472"""
"""ENSG00000079557"""
"""ENSG00000137078"""
"""ENSG00000186654"""
"""ENSG00000125458"""
…
"""ENSG00000107902"""
"""ENSG00000150995"""
"""ENSG00000178607"""
